# 02. Разведочный анализ данных (EDA) и исследование физико-химического пространства
**Магистерская диссертация:** «Разработка библиотеки бетавольтаических материалов» (СамГТУ)

Исследование корреляций между шириной запрещенной зоны $E_g$, плотностью $\rho$, порогом радиационных повреждений $E_d$ и термодинамической устойчивостью $E_{hull}$.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px

# Подключение к библиотеке SQLite
conn = sqlite3.connect('../data/05_database/betavoltaic_library.db')
df = pd.read_sql('''
    SELECT m.formula, m.crystal_system, m.material_class, m.density, m.is_viable,
           e.band_gap_dft, e.delta_eg_predicted, e.band_gap_calibrated, e.theoretical_efficiency_pct,
           p.ed_est_ev, p.radiation_resistance_score,
           p.penetration_depth_um_Ni63, p.carriers_per_electron_Ni63, p.t_max_ev_Ni63, p.is_immune_Ni63
    FROM materials m
    JOIN electronic_properties e ON m.mp_id = e.mp_id
    JOIN betavoltaic_performance p ON m.mp_id = p.mp_id
''', conn)
conn.close()

print(f'Всего загружено записей из библиотеки: {len(df):,}')
print(f'Жизнеспособных полупроводников (is_viable=1): {df["is_viable"].sum():,}')
df.head(10)

### Распределение жизнеспособных полупроводников по классам

In [ ]:
class_counts = df[df['is_viable'] == 1]['material_class'].value_counts()
class_counts

### Корреляция калиброванной запрещенной зоны и радиационной стойкости

In [ ]:
fig = px.scatter(
    df[df['is_viable'] == 1].sample(min(2000, len(df))),
    x='band_gap_calibrated',
    y='radiation_resistance_score',
    color='material_class',
    hover_name='formula',
    labels={'band_gap_calibrated': 'Eg калибр. (эВ)', 'radiation_resistance_score': 'Индекс стойкости R_score (%)'},
    title='Корреляция ширины зоны Eg и радиационной стойкости полупроводников'
)
fig.show()